# Retrieve and Clean Student CSV Datasets

This notebook loads the yearly student CSV files from `DataSets/`, standardizes their columns, combines them into a single dataset, and produces a cleaned version.

Run the cells in order from top to bottom. It assumes the notebook's working directory is `Scripts/`.

In [9]:
import csv
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd().resolve().parent
DATASETS_DIR = BASE_DIR / "DataSets"

## Helper Functions

Discover CSV files, detect their delimiter, and standardize column names.

In [2]:
def get_csv_files():
    csv_files = sorted(DATASETS_DIR.glob("*.csv"))

    if not csv_files:
        print(f"No CSV files found in {DATASETS_DIR}")
        return []

    return csv_files


def detect_delimiter(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        sample = file.read(4096)
        file.seek(0)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;")
        return dialect.delimiter
    except csv.Error:
        return ";" if ";" in sample else ","


def standardize_column_name(column_name):
    return str(column_name).strip().lower().replace(" ", "_")


def get_standardized_columns(csv_file: Path):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        header = next(reader, None)

    if header is None:
        return []

    standardized = [standardize_column_name(column) for column in header]
    return standardized


def load_csv_as_dataframe(csv_file: Path):
    delimiter = detect_delimiter(csv_file)
    df = pd.read_csv(csv_file, sep=delimiter, encoding="utf-8-sig")
    df.columns = [standardize_column_name(col) for col in df.columns]
    return df

## Preview Raw Files

Print the columns and first rows of each raw CSV file.

In [3]:
def print_first_rows(csv_file: Path, rows_to_show: int = 5):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        rows = list(reader)

    if not rows:
        print(f"\n=== {csv_file.name} ===")
        print("File is empty.")
        return

    print(f"\n=== {csv_file.name} ===")
    print("Columns:", get_standardized_columns(csv_file))
    for row in rows[:rows_to_show]:
        print(row)


csv_files = get_csv_files()

for csv_file in csv_files:
    print_first_rows(csv_file)


=== datos_estudiantes_2019.csv ===
Columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca']
['Id persona', 'Sexo', 'Rol', 'Departamento', 'Subsistema', 'Ciclo', 'Grado', 'Zona', 'Contexto', 'Año lectivo', 'Cantidad de días ingreso a CREA', 'Cantidad de entregas de tareas', 'Cantidad de Comentarios posteados', 'Cantidad de Acciones totales', 'Cantidad de días de ingreso a Matific', 'Cantidad de episodios finalizados en Matific', 'Cantidad de días de ingreso a PAM', 'Cantidad de actividades finalizadas en PAM', 'Cantidad de días de ingre

## Combine Datasets

Load every yearly CSV, standardize columns, concatenate, drop duplicates, and save the combined dataset.

> Note: the combined and cleaned outputs are large (500MB+) and are excluded from git via `.gitignore`.

In [4]:
def create_combined_dataset(output_name: str = "datos_estudiantes_total.csv"):
    csv_files = get_csv_files()
    if not csv_files:
        return None

    dataframes = [load_csv_as_dataframe(csv_file) for csv_file in csv_files]
    combined = pd.concat(dataframes, ignore_index=True)
    combined = combined.drop_duplicates()

    output_path = DATASETS_DIR / output_name
    combined.to_csv(output_path, index=False)

    print(f"Combined dataset saved to: {output_path}")
    print(f"Rows: {len(combined)}")
    print(f"Columns: {list(combined.columns)}")
    return combined


combined = create_combined_dataset()

Combined dataset saved to: /Users/gerardo/Documents/GitHub/PlanCeibal-UTEC26-MachineLearning/DataSets/datos_estudiantes_total.csv
Rows: 9120882
Columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca', 'cantidad_de_entregas_de_tareas_en_crea', 'cantidad_de_comentarios_posteados_en_crea', 'cantidad_de_acciones_totales_en_crea']


## Clean the Combined Dataset

Normalize text, replace known "missing" placeholders with NA, and coerce numeric-looking columns. Missing values are preserved so model preprocessing can be fit on training data only.

In [ ]:
def clean_combined_dataset(df: pd.DataFrame):
    cleaned = df.copy()
    cleaned = cleaned.drop_duplicates()

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            cleaned[column] = cleaned[column].astype(str).str.strip().str.lower()
            cleaned[column] = cleaned[column].replace(
                {
                    "": pd.NA,
                    "na": pd.NA,
                    "n/a": pd.NA,
                    "nan": pd.NA,
                    "null": pd.NA,
                    "none": pd.NA,
                    "sin dato": pd.NA,
                    "sin_dato": pd.NA,
                    "sin-dato": pd.NA,
                    "unknown": pd.NA,
                    "desconocido": pd.NA,
                }
            )

    if "id_persona" in cleaned.columns:
        cleaned = cleaned.dropna(subset=["id_persona"])

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
            valid_ratio = numeric_values.notna().sum() / max(cleaned[column].notna().sum(), 1)
            if valid_ratio > 0.8:
                cleaned[column] = numeric_values

    for column in cleaned.columns:
        if pd.api.types.is_numeric_dtype(cleaned[column]):
            if cleaned[column].dropna().mod(1).eq(0).all():
                cleaned[column] = cleaned[column].astype("Int64")

    return cleaned


if combined is not None:
    cleaned = clean_combined_dataset(combined)
    output_path = DATASETS_DIR / "datos_estudiantes_total_clean.csv"
    cleaned.to_csv(output_path, index=False)
    print(f"Clean dataset saved to: {output_path}")
    print(f"Cleaned rows: {len(cleaned)}")
    print(f"Cleaned columns: {list(cleaned.columns)}")

Clean dataset saved to: /Users/gerardo/Documents/GitHub/PlanCeibal-UTEC26-MachineLearning/DataSets/datos_estudiantes_total_clean.csv
Cleaned rows: 9120882
Cleaned columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca', 'cantidad_de_entregas_de_tareas_en_crea', 'cantidad_de_comentarios_posteados_en_crea', 'cantidad_de_acciones_totales_en_crea']


In [4]:
# BASE_DIR = Path(__file__).resolve().parent.parent
DATA_PATH = BASE_DIR / "DataSets" / "datos_estudiantes_total_clean.csv"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
RANDOM_STATE = 42
ACTIVITY_CANDIDATES = (
    "cantidad_de_acciones_totales_en_crea",
    "cantidad_de_acciones_totales",
    "cantidad_de_días_ingreso_a_crea",
    "cantidad_de_dias_ingreso_a_crea",
)


In [ ]:
def most_frequent(values: pd.Series):
    """Return a deterministic representative categorical value."""
    values = values.dropna()
    return values.mode().iloc[0] if not values.empty else pd.NA


def load_students() -> pd.DataFrame:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Create the cleaned dataset first: {DATA_PATH}")

    students = pd.read_csv(DATA_PATH, low_memory=False, dtype={"id_persona": "string"})
    required_columns = {"id_persona", "año_lectivo"}
    missing_columns = required_columns - set(students.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    students["id_persona"] = students["id_persona"].astype("string").str.strip()
    students["año_lectivo"] = pd.to_numeric(
        students["año_lectivo"], errors="coerce"
    ).astype("Int64")
    return students.dropna(subset=["id_persona", "año_lectivo"]).copy()


def choose_activity_column(students: pd.DataFrame) -> str:
    candidates = [column for column in ACTIVITY_CANDIDATES if column in students.columns]
    if not candidates:
        raise ValueError(
            "No supported CREA activity column found. Review ACTIVITY_CANDIDATES "
            "against the annual schema before training."
        )

    availability = pd.DataFrame(
        {
            column: students.groupby("año_lectivo")[column].apply(
                lambda values: pd.to_numeric(values, errors="coerce").notna().any()
            )
            for column in candidates
        }
    ).sort_index()
    print("CREA metric availability by year:")
    print(availability)

    canonical = next(
        (column for column in candidates if availability[column].all()), None
    )
    if canonical is None:
        raise ValueError(
            "No single CREA activity metric is observed in every year. Reconcile "
            "the annual schemas before training."
        )
    return canonical


def build_cohort(students: pd.DataFrame, activity_column: str) -> pd.DataFrame:
    students = students.copy()
    students[activity_column] = pd.to_numeric(
        students[activity_column], errors="coerce"
    )
    feature_columns = [
        column for column in students.columns if column not in {"id_persona", "año_lectivo"}
    ]
    numeric_columns = students[feature_columns].select_dtypes(include=np.number).columns
    categorical_columns = [column for column in feature_columns if column not in numeric_columns]
    aggregation = {column: "mean" for column in numeric_columns}
    aggregation.update({column: most_frequent for column in categorical_columns})

    student_year = students.groupby(["id_persona", "año_lectivo"], as_index=False).agg(
        aggregation
    )
    student_year["label_year"] = student_year["año_lectivo"] + 1
    next_year_activity = student_year[
        ["id_persona", "año_lectivo", activity_column]
    ].rename(
        columns={"año_lectivo": "label_year", activity_column: "next_year_crea_activity"}
    )
    cohort = student_year.merge(
        next_year_activity,
        on=["id_persona", "label_year"],
        how="inner",
        validate="many_to_one",
    ).rename(columns={"año_lectivo": "feature_year"})

    # An absent next-year activity measure is unknown, not evidence of zero activity.
    cohort = cohort.dropna(subset=["next_year_crea_activity"]).copy()
    cohort["engagement_risk"] = (cohort["next_year_crea_activity"] == 0).astype(int)
    return cohort


def build_pipeline(train: pd.DataFrame, model_features: list[str]) -> Pipeline:
    numeric_features = [
        column for column in model_features if pd.api.types.is_numeric_dtype(train[column])
    ]
    categorical_features = [column for column in model_features if column not in numeric_features]
    transformers = []
    if numeric_features:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            )
        )
    if categorical_features:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            )
        )
    if not transformers:
        raise ValueError("No usable model features remain.")

    return Pipeline(
        [
            ("preprocessor", ColumnTransformer(transformers)),
            (
                "classifier",
                LogisticRegression(
                    max_iter=500, class_weight="balanced", random_state=RANDOM_STATE
                ),
            ),
        ]
    )

def select_threshold(y_true: pd.Series, probabilities: np.ndarray) -> float:
    candidates = np.arange(0.05, 0.96, 0.05)
    return max(
        candidates, key=lambda threshold: f1_score(y_true, probabilities >= threshold)
    )



In [ ]:
def main() -> None:
    students = load_students()
    print("Rows and unique students by year:")
    print(
        students.groupby("año_lectivo")["id_persona"]
        .agg(rows="size", unique_students="nunique")
        .sort_index()
    )
    print(
        "Duplicate student-year rows:",
        students.duplicated(["id_persona", "año_lectivo"]).sum(),
    )

    activity_column = choose_activity_column(students)
    cohort = build_cohort(students, activity_column)
    print("Risk rate by feature year:")
    print(cohort.groupby("feature_year")["engagement_risk"].agg(["size", "mean"]))

    cohort_years = sorted(cohort["feature_year"].unique())
    if len(cohort_years) < 3:
        raise ValueEsrror(f"Need three valid feature years; found {cohort_years}")
    train_years, validation_year, test_year = cohort_years[:-2], cohort_years[-2], cohort_years[-1]
    excluded = {
        "id_persona",
        "feature_year",
        "label_year",
        "next_year_crea_activity",
        "engagement_risk",
    }
    model_features = [column for column in cohort.columns if column not in excluded]

    train = cohort[cohort["feature_year"].isin(train_years)].copy()
    validation = cohort[cohort["feature_year"] == validation_year].copy()
    test = cohort[cohort["feature_year"] == test_year].copy()

    for name, frame in {"train": train, "validation": validation, "test": test}.items():
        if frame["engagement_risk"].nunique() < 2:
            raise ValueError(f"{name} cohort has only one target class.")

    model = build_pipeline(train, model_features)

    x_train, y_train = train[model_features], train["engagement_risk"]
    x_validation, y_validation = validation[model_features], validation["engagement_risk"]
    x_test, y_test = test[model_features], test["engagement_risk"]

    model.fit(x_train, y_train)
    dummy = DummyClassifier(strategy="prior", random_state=RANDOM_STATE).fit(x_train, y_train)

    threshold = select_threshold(y_validation, model.predict_proba(x_validation)[:, 1])
    probabilities = model.predict_proba(x_test)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    dummy_probabilities = dummy.predict_proba(x_test)[:, 1]
    metrics = {
        "train_years": [int(year) for year in train_years],
        "validation_year": int(validation_year),
        "test_year": int(test_year),
        "threshold": float(threshold),
        "test_rows": int(len(test)),
        "test_risk_rate": float(y_test.mean()),
        "roc_auc": float(roc_auc_score(y_test, probabilities)),
        "pr_auc": float(average_precision_score(y_test, probabilities)),
        "precision": float(precision_score(y_test, predictions, zero_division=0)),
        "recall": float(recall_score(y_test, predictions, zero_division=0)),
        "f1": float(f1_score(y_test, predictions, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, predictions)),
        "dummy_pr_auc": float(average_precision_score(y_test, dummy_probabilities)),
        "activity_column": activity_column,
    }
    print(json.dumps(metrics, indent=2))

    ARTIFACTS_DIR.mkdir(exist_ok=True)
    dump(model, ARTIFACTS_DIR / "engagement_risk_pipeline.joblib")
    (ARTIFACTS_DIR / "engagement_risk_metrics.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )
    (ARTIFACTS_DIR / "engagement_risk_features.json").write_text(
        json.dumps(model_features, indent=2), encoding="utf-8"
    )


if __name__ == "__main__":
    main()


Rows and unique students by year:
                rows  unique_students
año_lectivo                          
2019         1336094           668047
2020         1336220           668074
2021         1333950           666915
2022         1315412           657667
2023         1294752           647139
2024         1263918           631828
2025         1240536           619819
Duplicate student-year rows: 4561393
CREA metric availability by year:
             cantidad_de_acciones_totales_en_crea  \
año_lectivo                                         
2019                                         True   
2020                                         True   
2021                                         True   
2022                                         True   
2023                                         True   
2024                                         True   
2025                                         True   

             cantidad_de_acciones_totales  cantidad_de_días_ingreso_a_crea 

ValueError: train cohort has only one target class.